# Market Share by Brand & Territory (Q4-2021)

Write a query to find the Market Share at the Product Brand level for each Territory, for Time Period Q4-2021. Market Share is the number of Products of a certain Product Brand brand sold in a territory, divided by the total number of Products sold in this Territory.

Output the ID of the Territory, name of the Product Brand and the corresponding Market Share in percentages. Only include these Product Brands that had at least one sale in a given territory.

🌀By solving this, you'll learn how to use Mutiple Cte, Windows Function, Group by, Join. Give it a try and share the output! 👇

In [0]:

CREATE TABLE ska_catalog2.bronze.fct_customer_sales (cust_id VARCHAR(50), prod_sku_id VARCHAR(50), order_date DATE, order_value BIGINT, order_id VARCHAR(50));

CREATE TABLE ska_catalog2.bronze.map_customer_territory (cust_id VARCHAR(50), territory_id VARCHAR(50));

CREATE TABLE ska_catalog2.bronze.dim_product (prod_sku_id VARCHAR(50), prod_sku_name VARCHAR(255), prod_brand VARCHAR(100), market_name VARCHAR(100));

INSERT INTO ska_catalog2.bronze.fct_customer_sales (cust_id, prod_sku_id, order_date, order_value, order_id) VALUES ('C001', 'P001', '2021-10-15', 100, 'O1001'), ('C002', 'P002', '2021-11-20', 200, 'O1002'), ('C003', 'P003', '2021-12-05', 150, 'O1003'), ('C001', 'P002', '2021-12-10', 300, 'O1004'), ('C002', 'P001', '2021-11-18', 250, 'O1005');

INSERT INTO ska_catalog2.bronze.map_customer_territory (cust_id, territory_id) VALUES ('C001', 'T001'), ('C002', 'T002'), ('C003', 'T001');

INSERT INTO ska_catalog2.bronze.dim_product (prod_sku_id, prod_sku_name, prod_brand, market_name) VALUES ('P001', 'Product A', 'Brand X', 'Market 1'), ('P002', 'Product B', 'Brand Y', 'Market 2'), ('P003', 'Product C', 'Brand X', 'Market 1');

In [0]:
SELECT * FROM ska_catalog2.bronze.fct_customer_sales;

In [0]:
SELECT * FROM ska_catalog2.bronze.map_customer_territory;

In [0]:
SELECT * FROM ska_catalog2.bronze.dim_product;

In [0]:
%sql
WITH sales_data AS (
  SELECT 
    mct.territory_id,
    dp.prod_brand,
    COUNT(fcs.prod_sku_id) AS brand_sales
  FROM ska_catalog2.bronze.fct_customer_sales fcs
  JOIN ska_catalog2.bronze.map_customer_territory mct ON fcs.cust_id = mct.cust_id
  JOIN ska_catalog2.bronze.dim_product dp ON fcs.prod_sku_id = dp.prod_sku_id
  WHERE fcs.order_date > '2021-10-01' AND fcs.order_date < '2022-01-01'
  GROUP BY mct.territory_id, dp.prod_brand
),
total_sales_per_territory AS (
  SELECT
    territory_id,
    SUM(brand_sales) AS `total_sales`
  FROM sales_data
  GROUP BY territory_id
)
SELECT 
  s.territory_id,
  s.prod_brand,
  ROUND((CAST(s.brand_sales AS FLOAT)/ t.total_sales)*100,2) AS `market_share_percentage`
FROM sales_data s
JOIN total_sales_per_territory t ON s.territory_id = t.territory_id
ORDER BY s.territory_id, `market_share_percentage` DESC;

Databricks visualization. Run in Databricks to view.